[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch02-ml-systems/02-khoang-cach-bang-thong-nang-luong.ipynb)

# Khoảng cách, băng thông và năng lượng: khi nào phải tính tại chỗ

Ràng buộc thứ nhất của chương là rào cản ánh sáng: không có quy định hay mô hình kinh tế nào làm ánh sáng
nhanh hơn. Nhưng chương cũng nói đặc trưng định nghĩa của edge không hẳn là xử lý ở đâu, mà là nơi đó phải
gánh bao nhiêu dữ liệu. Sổ tay này tính cả hai mặt đó: thời gian để dữ liệu đi xa, và cái giá để dữ liệu
đi, bằng mi-li-giây, bằng byte mỗi giây và bằng jun.

**Bạn sẽ làm:**
1. tính mức sàn độ trễ (latency) do ánh sáng trong sợi quang, và khoảng cách mà mỗi ngân sách độ trễ cho
   phép;
2. áp bất biến cục bộ dữ liệu cho drone tránh vật cản 4K của chương, rồi tìm cỡ dữ liệu tối đa mà một
   đường xa còn kịp hạn chót;
3. tính điểm nghẽn băng thông (bandwidth) của 100 camera trong nhà máy, và mức giảm khi chỉ gửi siêu dữ liệu;
4. so năng lượng truyền dữ liệu với năng lượng suy luận (inference) tại chỗ, và tìm cỡ dữ liệu hoà vốn;
5. tính bức tường của trợ lý giọng nói khi mọi truy vấn đều lên cloud.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Rào cản ánh sáng

Thời gian một vòng tối thiểu bằng hai lần khoảng cách một chiều chia cho tốc độ ánh sáng trong sợi quang,
khoảng 200,000 km/s, tức chỉ khoảng hai phần ba tốc độ trong chân không:

$$L_\text{lat,min} = \frac{2 \times \text{khoảng cách}}{c_\text{fiber}}$$

Đó là mức sàn trước khi bắt đầu bất kỳ phép tính nào. Dịch vụ thật còn cộng thêm định tuyến, giao thức,
xếp hàng và thời gian xử lý.

In [ ]:
MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định, như các sổ tay khác
C_SOI_QUANG = sach(200_000, nguon="200,000 km/s")    # km/s


def vong_toi_thieu_ms(km_mot_chieu):
    """Thời gian một vòng tối thiểu qua sợi quang, chưa tính bất kỳ phép tính nào."""
    return 2 * km_mot_chieu / C_SOI_QUANG * 1e3


def km_toi_da(ngan_sach_ms):
    """Khoảng cách một chiều xa nhất mà riêng ánh sáng đã dùng hết ngân sách."""
    return ngan_sach_ms / 1e3 * C_SOI_QUANG / 2


CA_VA = sach(3600, nguon="3,600 km")
theo_sach("California → Virginia, một vòng (ms)", vong_toi_thieu_ms(CA_VA), sach=36, nguon="36 ms")

# Napkin Math 1.3: bộ giám sát an toàn cho cánh tay robot, mô hình chạy ở trung tâm dữ liệu cách 1,500 km
ROBOT_KM = sach(1500, nguon="1,500 km, 15 ms")
NGAN_SACH_ROBOT = sach(10, trich="requires a 10 ms end-to-end response time")
theo_sach("cánh tay robot, một vòng (ms)", vong_toi_thieu_ms(ROBOT_KM), sach=15, nguon="1,500 km, 15 ms")
theo_sach("dư địa so với ngân sách (ms)", NGAN_SACH_ROBOT - vong_toi_thieu_ms(ROBOT_KM), sach=-5,
          trich="(-5 ms headroom)")

### Dự đoán

Một hệ thống phanh an toàn có ngân sách 10 ms. Giả sử mô hình ở xa chạy mất đúng 0 ms. Trung tâm dữ liệu
có thể đặt xa nhất là bao nhiêu ki-lô-mét? Còn một khung hình thực tại tăng cường ở 60 FPS, với ngân sách
16.7 ms?

In [ ]:
PHANH = sach(10, nguon="10 ms")
KHUNG_AR = sach(16.7, nguon="16.7 ms")
for ten, ms in (("phanh an toàn", PHANH), ("một khung AR ở 60 FPS", KHUNG_AR)):
    print(f"{ten:<22} {ms:>5} ms → trung tâm dữ liệu xa nhất {km_toi_da(ms):>6,.0f} km, nếu mô hình chạy mất 0 ms")

# Bảng độ trễ của chương: nếu toàn bộ thời gian chỉ là ánh sáng đi trong sợi quang, khoảng cách là bao xa?
MANG = {"cùng trung tâm dữ liệu": (sach(0.5, nguon="0.5 / 1–5 / 50–150 ms"),) * 2,
        "cùng vùng": (sach(1, nguon="0.5 / 1–5 / 50–150 ms"), sach(5, nguon="0.5 / 1–5 / 50–150 ms")),
        "khác vùng": (sach(50, nguon="0.5 / 1–5 / 50–150 ms"), sach(150, nguon="0.5 / 1–5 / 50–150 ms"))}
for ten, (thap, cao) in MANG.items():
    ms = f"{thap}" if thap == cao else f"{thap}–{cao}"
    km_txt = f"{km_toi_da(thap):,.0f}" if thap == cao else f"{km_toi_da(thap):,.0f}–{km_toi_da(cao):,.0f}"
    print(f"{ten:<22} {ms:>6} ms → nếu chỉ có ánh sáng: tối đa {km_txt} km")

In [ ]:
km = np.linspace(0, 4000, 200)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(km, vong_toi_thieu_ms(km), color=MAU[0], lw=2, label="mức sàn: một vòng qua sợi quang")
for ms, ten, mau in ((PHANH, "phanh an toàn", MAU[1]), (KHUNG_AR, "một khung AR ở 60 FPS", MAU[3])):
    ax.axhline(ms, color=mau, lw=1.2, ls="--")
    ax.text(3950, ms + 0.8, f"{ten}: {ms} ms", ha="right", fontsize=8, color="0.25")
    ax.fill_between(km, ms, vong_toi_thieu_ms(km), where=vong_toi_thieu_ms(km) > ms, color=mau, alpha=0.08, lw=0)
for d, ten, cho_chu in ((ROBOT_KM, "cánh tay robot", (1750, 5)), (CA_VA, "California → Virginia", (1500, 38))):
    ax.plot([d], [vong_toi_thieu_ms(d)], "o", color=MAU[0], ms=8, mec="white", mew=1.5)
    ax.annotate(f"{ten}: {d:,} km, {vong_toi_thieu_ms(d):.0f} ms", (d, vong_toi_thieu_ms(d)), xytext=cho_chu,
                fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xlim(0, 4000)
ax.set_ylim(0, 42)
ax.set_xlabel("khoảng cách một chiều (km)")
ax.set_ylabel("thời gian một vòng tối thiểu (ms)")
ax.set_title("Vùng tô màu: riêng ánh sáng đã vượt ngân sách")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Với 10 ms, trung tâm dữ liệu phải nằm trong vòng 1,000 km, và đó là khi mạng không định tuyến vòng, không
xếp hàng, và mô hình không tốn chút thời gian nào. Các con số mạng trong bảng độ trễ của chương cho thấy thực tế xa mức sàn
này: một lệnh gọi khác vùng mất 50–150 ms, ứng với hàng nghìn ki-lô-mét nếu chỉ tính ánh sáng.

Vì thế chương nói một mô hình cloud đúng 99 % vẫn vô dụng cho phanh khẩn cấp tự động nếu độ trễ mạng vượt
thời gian phản ứng cho phép. Cổng độ trễ loại cloud trước khi cần xét tới tính toán hay chi phí.

## 2. Bất biến cục bộ dữ liệu

Chương so hai đường đi hoàn chỉnh với nhau, và với hạn chót của dịch vụ:

$$T_\text{remote} = \frac{D_\text{vol}}{\text{BW}_\text{network}} + L_\text{lat,network} + \frac{O}{R_\text{peak,remote}\cdot\eta_\text{hw,remote}},
\qquad \text{chạy ở xa khả thi} \iff T_\text{remote} \le T_\text{deadline}$$

Nếu cả hai đường đều kịp hạn, đường nhanh hơn được chọn bằng cách so tổng thời gian cục bộ với tổng thời
gian ở xa.

Napkin Math 1.7 xét một drone tránh vật cản quay 4K ở 60 FPS, gửi từng khung qua đường truyền lên 100 Mb/s
của mạng gia đình, với 110 ms cho vòng mạng và phần tính ở xa. Sổ tay **giả định** một khung 4K là
3840×2160 điểm ảnh, 3 byte mỗi điểm, đúng cách Napkin Math 1.6 đếm một khung 1080p.

**Dự đoán:** nếu mô hình ở xa nhanh vô hạn, đường xa có kịp một khung hình không?

In [ ]:
FPS_DRONE = sach(60, trich="(4K, 60 FPS)")
byte_4k = 3840 * 2160 * 3
theo_sach("một khung 4K (MB)", byte_4k / 1e6, sach=24.9, trich="4K frame ≈ 24.9 MB", tol=0.05)
BW_NHA = sach(100, trich="100 Mb/s home broadband (up)") * 1e6     # bit mỗi giây
PHAN_HOI_XA = sach(110, trich="110 ms (round-trip + remote compute)")
KHUNG_MS = sach(16.7, trich="vs. a 16.7 ms frame interval")


def duong_xa_ms(byte, bit_moi_giay, phan_hoi_ms):
    """Toàn bộ đường xa: truyền dữ liệu lên, rồi vòng mạng cộng phần tính ở xa."""
    return byte * 8 / bit_moi_giay * 1e3 + phan_hoi_ms


truyen_ms = byte_4k * 8 / BW_NHA * 1e3
theo_sach("truyền một khung (ms)", truyen_ms, sach=1990.7, trich="1,990.7 ms")
theo_sach("toàn bộ đường xa (ms)", duong_xa_ms(byte_4k, BW_NHA, PHAN_HOI_XA), sach=2100.7, trich="2100.7 ms")
theo_sach("khoảng cách giữa hai khung ở 60 FPS (ms)", 1000 / FPS_DRONE, sach=16.7, trich="vs. a 16.7 ms frame interval")
print(f"đường xa dài gấp {duong_xa_ms(byte_4k, BW_NHA, PHAN_HOI_XA) / KHUNG_MS:.0f} lần một khung hình; "
      f"riêng phần cố định 110 ms đã gấp {PHAN_HOI_XA / KHUNG_MS:.1f} lần")

Đường xa trượt hạn chót trước cả khi tính tới độ trễ còn lại của vòng điều khiển, và tính toán ở xa
nhanh hơn cũng không xoá được thời gian truyền. Thậm chí một dữ liệu bé tí cũng không cứu được nó: riêng
110 ms cố định đã dài hơn một khung hình.

Phần A của lab đi kèm chương đặt câu hỏi ngược lại: với một hạn chót và một đường truyền, dữ liệu lớn tới
đâu thì đường xa vẫn còn kịp? Từ công thức trên, cỡ dữ liệu tối đa là
$(T_\text{deadline} - L)\cdot \text{BW}_\text{network}$, với $L$ là toàn bộ phần cố định: vòng mạng cộng
phần tính ở xa. Bảng dưới thử hai phần cố định: 110 ms của drone, và 5 ms, mức cao của một lệnh gọi cùng
vùng trong bảng độ trễ, với **giả định** phần tính ở xa mất 0 ms. Các hạn chót là một khung hình, mức
"dưới 100 ms" mà chương gắn cho edge, và 1 giây, một hạn chót do sổ tay **giả định**.

In [ ]:
DUOI_100 = sach(100, trich="sub-100 ms latency")


def byte_toi_da(han_ms, bit_moi_giay, phan_hoi_ms):
    """Dữ liệu lớn nhất mà đường xa còn kịp hạn chót; 0 nếu riêng phần cố định đã trễ."""
    return max(0.0, (han_ms - phan_hoi_ms) / 1e3 * bit_moi_giay / 8)


def co(b):
    """Viết một cỡ dữ liệu cho dễ đọc."""
    if b == 0:
        return "không kịp"
    return f"{b / 1e3:,.1f} KB" if b < 1e6 else f"{b / 1e6:,.2f} MB"


CUNG_VUNG = MANG["cùng vùng"][1]      # ms, mức cao của một lệnh gọi cùng vùng
cac_duong = {"10 Mb/s": 10e6, "100 Mb/s": BW_NHA, "1 Gb/s": 1e9}
print(f"{'hạn chót':>9} | {'cố định':>8} | " + " | ".join(f"{k:>10}" for k in cac_duong))
cho_duoc_4k = []
for han, L in ((KHUNG_MS, PHAN_HOI_XA), (DUOI_100, PHAN_HOI_XA), (1000, PHAN_HOI_XA),
               (KHUNG_MS, CUNG_VUNG), (DUOI_100, CUNG_VUNG)):
    o = {k: byte_toi_da(han, bw, L) for k, bw in cac_duong.items()}
    print(f"{han:>6} ms | {L:>5} ms | " + " | ".join(f"{co(b):>10}" for b in o.values()))
    cho_duoc_4k += [f"hạn {han} ms, cố định {L} ms, {k}" for k, b in o.items() if b >= byte_4k]
print("chở kịp một khung 4K:", "; ".join(cho_duoc_4k) or "không ô nào")
assert byte_toi_da(KHUNG_MS, 1e12, PHAN_HOI_XA) == 0   # đường truyền nhanh đến đâu cũng không kịp một khung

In [ ]:
cac_byte = np.logspace(3, 8, 200)   # 1 KB tới 100 MB
fig, ax = plt.subplots(figsize=(8, 4.4))
for (ten, bw), mau in zip({"10 Mb/s": 10e6, "100 Mb/s": BW_NHA, "1 Gb/s": 1e9, "10 Gb/s": 10e9}.items(), MAU):
    ax.loglog(cac_byte / 1e6, duong_xa_ms(cac_byte, bw, PHAN_HOI_XA), color=mau, lw=2, label=f"đường truyền {ten}")
ax.axhline(KHUNG_MS, color="0.3", lw=1, ls="--")
ax.text(1.2e-3, KHUNG_MS * 1.15, f"một khung ở 60 FPS: {KHUNG_MS} ms", fontsize=8, color="0.25")
ax.axhline(PHAN_HOI_XA, color="0.6", lw=0.8, ls=":")
ax.text(1.2e-3, PHAN_HOI_XA * 0.72, f"phần cố định: {PHAN_HOI_XA} ms", fontsize=8, color="0.35")
ax.plot([byte_4k / 1e6], [duong_xa_ms(byte_4k, BW_NHA, PHAN_HOI_XA)], "o", color=MAU[1], ms=8, mec="white", mew=1.5)
ax.annotate(f"khung 4K qua 100 Mb/s: {duong_xa_ms(byte_4k, BW_NHA, PHAN_HOI_XA):,.0f} ms", (byte_4k / 1e6,
            duong_xa_ms(byte_4k, BW_NHA, PHAN_HOI_XA)), xytext=(-190, 12), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(8, 2e5)
ax.set_xlabel("dữ liệu gửi đi mỗi khung (MB)")
ax.set_ylabel("toàn bộ đường xa (ms)")
ax.set_title("Đường xa của drone: không đường truyền nào xuống dưới một khung hình")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Bên trái, mọi đường cong nằm sát phần cố định 110 ms: với dữ liệu nhỏ, chính vòng mạng quyết định. Bên
phải, thời gian truyền chiếm hết, và đường truyền nhanh gấp 10 lần thì nhanh gấp 10 lần. Không có chỗ nào
đường xa xuống dưới 16.7 ms, nên drone phải tính tại chỗ. Bảng ở trên cũng cho thấy: kể cả khi phần cố
định chỉ còn 5 ms, trong 100 ms đường 1 Gb/s chở chưa tới nửa khung 4K.

Chương vẫn nhắc: hạn chót loại được cả hai đường, nên đường cục bộ cũng phải kịp nó, chứ không tự động
thắng.

## 3. Điểm nghẽn băng thông: 100 camera trong nhà máy

Napkin Math 1.6 xét một hệ thống kiểm tra chất lượng có 100 camera 1080p chạy ở 30 FPS. Câu hỏi không
còn là bao lâu, mà là đường truyền có chở nổi không.

**Dự đoán:** một đường truyền riêng 10 Gbps chở được bao nhiêu camera như thế, gửi ảnh thô?

In [ ]:
RONG = sach(1920, trich=r"1920 \(\times\) 1080 \(\times\) 3 bytes \(\times\) 30 FPS")
CAO = sach(1080, trich=r"1920 \(\times\) 1080 \(\times\) 3 bytes \(\times\) 30 FPS")
BYTE_DIEM = sach(3, trich=r"1920 \(\times\) 1080 \(\times\) 3 bytes \(\times\) 30 FPS")
FPS_CAM = sach(30, trich=r"1920 \(\times\) 1080 \(\times\) 3 bytes \(\times\) 30 FPS")
SO_CAM = sach(100, trich="100 cameras running at 30 FPS")
mot_cam = RONG * CAO * BYTE_DIEM * FPS_CAM            # byte mỗi giây
ca_san = SO_CAM * mot_cam
theo_sach("một camera (MB/s)", mot_cam / 1e6, sach=186.6, trich="≈ 186.6 MB/s", tol=0.05)
theo_sach("cả sàn nhà máy (GB/s)", ca_san / 1e9, sach=18.7, trich="= 18.7 GB/s")

DUONG_RIENG = sach(10, trich="dedicated 10 Gbps line (1.25 GB/s)") * 1e9 / 8   # byte mỗi giây
theo_sach("đường truyền 10 Gbps (GB/s)", DUONG_RIENG / 1e9, sach=1.25, trich="dedicated 10 Gbps line (1.25 GB/s)")
theo_sach("tải cần nhiều băng thông hơn (lần)", ca_san / DUONG_RIENG, sach=14.9, trich="14.9× more bandwidth")
print(f"đường truyền đó chở được {math.floor(DUONG_RIENG / mot_cam)} camera ảnh thô")

# phí chuyển dữ liệu ra, nếu dòng thô sau này được lấy ra khỏi cloud; giả định một tháng là 365/12 ngày
GIA_RA = sach(0.09, trich="$0.09/GB data-transfer-out pricing")
gb_moi_thang = ca_san * 86_400 * 365 / 12 / 1e9
theo_sach("phí chuyển ra mỗi tháng (triệu đô la)", gb_moi_thang * GIA_RA / 1e6, sach=4.4, trich="$4.4M/month")

Chương kết luận vật lý đã làm việc truyền thô lên cloud thành bất khả thi cho ứng dụng này, nên xử lý ở
edge là bắt buộc. Nếu máy chủ edge chỉ gửi siêu dữ liệu về lỗi, khoảng 1 KB cho mỗi lần phát hiện và
khoảng 20 lần mỗi giây trên cả sàn, lưu lượng giảm bao nhiêu? Đây cũng là ý của phần D trong lab: lọc tại
chỗ, chỉ gửi đi phần đáng gửi.

In [ ]:
KB_LAN = sach(1, trich="1 KB per detection at roughly 20 events/s") * 1_000
LAN_GIAY = sach(20, trich="1 KB per detection at roughly 20 events/s")
sieu_du_lieu = KB_LAN * LAN_GIAY
theo_sach("mức giảm khi chỉ gửi siêu dữ liệu (lần)", ca_san / sieu_du_lieu, sach=933_120, trich="933,120×")
print(f"siêu dữ liệu: {sieu_du_lieu / 1e3:.0f} KB/s; đường 10 Gbps khi đó chở được {DUONG_RIENG / sieu_du_lieu:,.0f} sàn như vậy")

luong = {"một camera, ảnh thô": mot_cam, "100 camera, ảnh thô": ca_san,
         "đường truyền 10 Gbps": DUONG_RIENG, "chỉ siêu dữ liệu": sieu_du_lieu}
fig, ax = plt.subplots(figsize=(8, 3.2))
mau_thanh = [MAU[0], MAU[0], "0.55", MAU[2]]
ax.barh(list(luong), list(luong.values()), color=mau_thanh, height=0.6)
def toc_do(b):
    """Viết một tốc độ dữ liệu theo đơn vị dễ đọc."""
    for chia, don_vi in ((1e9, "GB/s"), (1e6, "MB/s"), (1e3, "KB/s")):
        if b >= chia:
            return f"{b / chia:,.3g} {don_vi}"
    return f"{b:,.0f} B/s"


for i, v in enumerate(luong.values()):
    ax.text(v * 1.3, i, toc_do(v), va="center", fontsize=8)
ax.set_xscale("log")
ax.set_xlim(5e3, 2e11)
ax.invert_yaxis()
ax.set_xlabel("byte mỗi giây (thang log)")
ax.set_title("Dữ liệu sinh ra, đường truyền có, và dữ liệu đáng gửi")
plt.tight_layout()
plt.show()

## 4. Năng lượng của việc truyền

Với thiết bị chạy pin, mi-li-giây chưa đủ, phải tính cả jun. Napkin Math 1.1 so hai cách cho một cảm biến:
gửi 1 MB dữ liệu qua Wi-Fi hoặc LTE với khoảng 100 mJ mỗi MB, hay chạy một lần suy luận trên NPU tại chỗ,
khoảng 0.1 mJ. Cả hai con số đều là ví dụ minh hoạ của chương.

**Dự đoán:** cảm biến phải gửi ít dữ liệu tới mức nào thì việc gửi mới rẻ hơn tự tính?

In [ ]:
D_MB = sach(1, trich="1 MB (illustrative payload volume)")
E_TRUYEN = sach(100, trich="100 mJ/MB (Wi-Fi/LTE)")       # mJ mỗi MB
E_SUY_LUAN = sach(0.1, trich="0.1 mJ/inference")          # mJ mỗi lần
e_cloud = D_MB * E_TRUYEN
theo_sach("gửi lên cloud (mJ)", e_cloud, sach=100, trich="100 mJ/MB (Wi-Fi/LTE)")
theo_sach("truyền đắt hơn tính tại chỗ (lần)", e_cloud / E_SUY_LUAN, sach=1000, nguon="1,000×")
hoa_von_kb = E_SUY_LUAN / E_TRUYEN * 1000
print(f"cỡ dữ liệu hoà vốn: {hoa_von_kb:.1f} KB; gửi ít hơn thế mới rẻ hơn một lần suy luận tại chỗ")

PIN_DIEN_THOAI = sach(15, nguon="15 Wh–22 Wh") * 3600      # jun
print(f"pin {PIN_DIEN_THOAI / 3600:.0f} Wh: đủ cho {PIN_DIEN_THOAI / (e_cloud / 1e3):,.0f} lần gửi 1 MB, "
      f"hoặc {PIN_DIEN_THOAI / (E_SUY_LUAN / 1e3):,.0f} lần suy luận tại chỗ (chỉ tính hai khoản này)")

In [ ]:
cac_mb = np.logspace(-4, 1, 200)   # 100 byte tới 10 MB
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.loglog(cac_mb * 1000, cac_mb * E_TRUYEN, color=MAU[1], lw=2, label="gửi dữ liệu thô, 100 mJ mỗi MB")
ax.axhline(E_SUY_LUAN, color=MAU[0], lw=2, label="một lần suy luận tại chỗ, 0.1 mJ")
ax.plot([hoa_von_kb], [E_SUY_LUAN], "o", color="0.3", ms=7)
ax.annotate(f"hoà vốn ở {hoa_von_kb:.0f} KB", (hoa_von_kb, E_SUY_LUAN), xytext=(-95, 18), textcoords="offset points",
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.plot([D_MB * 1000], [e_cloud], "o", color=MAU[1], ms=8, mec="white", mew=1.5)
ax.annotate("1 MB: 1,000 lần một lần suy luận", (D_MB * 1000, e_cloud), xytext=(-210, 4), textcoords="offset points",
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xlabel("dữ liệu gửi đi (KB)")
ax.set_ylabel("năng lượng (mJ)")
ax.set_title("Napkin Math 1.1: truyền hay tính tại chỗ")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Theo các giả định đó, gửi dữ liệu thô chỉ rẻ hơn khi nó nhỏ hơn khoảng 1 KB, cỡ một bản tóm tắt chứ không
phải một khung hình. Chương lưu ý con số 1,000× phụ thuộc kích thước dữ liệu, loại radio, chu kỳ hoạt động
(duty cycle) và sức tính toán cục bộ.

### Xử lý phân tầng: cảm biến động vật hoang dã

Trong bảo tồn động vật hoang dã, cảm biến âm thanh xử lý dòng âm thanh ngay tại chỗ. Với giả định minh hoạ
của chương, lượng truyền giảm từ 4.3 GB mỗi ngày âm thanh thô xuống 400 KB mỗi ngày bản tóm tắt. Nếu radio
của cảm biến tốn đúng 100 mJ mỗi MB như ở trên (một **giả định** của sổ tay, vì chương không cho radio của
cảm biến này), công suất trung bình chỉ riêng cho radio là bao nhiêu?

In [ ]:
THO_GB = sach(4.3, nguon="4.3 GB/ngày → 400 KB/ngày, 10,750×")
TOM_TAT_KB = sach(400, nguon="4.3 GB/ngày → 400 KB/ngày, 10,750×")
theo_sach("mức giảm truyền (lần)", THO_GB * 1e9 / (TOM_TAT_KB * 1e3), sach=10_750, nguon="4.3 GB/ngày → 400 KB/ngày, 10,750×")
NGUONG_TINY = sach(1, nguon="<1 TOPS, <1 mW")   # mW
for ten, mb_ngay in (("âm thanh thô", THO_GB * 1000), ("bản tóm tắt", TOM_TAT_KB / 1000)):
    jun_ngay = mb_ngay * E_TRUYEN / 1e3
    mw = jun_ngay / 86_400 * 1e3
    print(f"{ten:<13}: {jun_ngay:9.3f} J mỗi ngày cho radio, trung bình {mw:.3g} mW "
          f"({mw / NGUONG_TINY:.3g} lần mức dưới {NGUONG_TINY} mW của TinyML)")

Gửi âm thanh thô sẽ tiêu riêng cho radio gấp vài lần toàn bộ ngân sách công suất của một thiết bị TinyML.
Gửi bản tóm tắt thì phần radio gần như biến mất. Đó là mẫu xử lý phân tầng của chương: tầng dưới cùng lọc,
tầng trên chỉ nhận phần đáng nhận, và mẫu này được chọn khi lượng dữ liệu vượt năng lực truyền.

## 5. Bức tường của trợ lý giọng nói

Napkin Math 1.5 là một kịch bản năng lực minh hoạ: 1 tỉ thiết bị, mỗi thiết bị 20 truy vấn mỗi ngày. Nếu
mọi truy vấn đều được phục vụ trên cloud, mỗi truy vấn tốn khoảng 200 ms thời gian GPU, và một trung tâm
dữ liệu trong kịch bản có 10,000 GPU, thì cần bao nhiêu trung tâm dữ liệu?

**Dự đoán:** một, năm, hay hai mươi?

In [ ]:
THIET_BI = sach(1, trich=r"1 billion devices \(\times\) 20 queries/day") * 1e9
TRUY_VAN = sach(20, trich="20 queries/day")
GPU_GIAY = sach(200, trich="~200 ms of GPU time", tol=0.5) / 1e3
GPU_MOI_TT = sach(10_000, trich="10,000 GPUs")
DINH = sach(4.5, trich="~4.5× peak-to-average", tol=0.05)

truy_van_ngay = THIET_BI * TRUY_VAN
theo_sach("truy vấn mỗi ngày", truy_van_ngay, sach=20e9, trich="20B queries/day")
gpu_gio = truy_van_ngay * GPU_GIAY / 3600
theo_sach("giờ GPU mỗi ngày", gpu_gio, sach=1_111_111, trich="1,111,111 GPU-hours/day")
tt_gio = GPU_MOI_TT * 24
theo_sach("giờ GPU một trung tâm dữ liệu có mỗi ngày", tt_gio, sach=240_000, trich="240,000 GPU-hours/day")
theo_sach("trung tâm dữ liệu, trung bình", gpu_gio / tt_gio, sach=4.6, trich="~4.6 dedicated data centers", tol=0.05)
theo_sach("trung tâm dữ liệu, giờ cao điểm", DINH * gpu_gio / tt_gio, sach=20.8, trich="~20.8 data centers at peak", tol=0.05)

CHI_PHI_TB = sach(0.5, trich="~$0.50/device/year", tol=0.005)
theo_sach("chi phí cloud mỗi năm (đô la)", CHI_PHI_TB * THIET_BI, sach=500_000_000, trich="$500,000,000/year")

# phản thực: nếu mọi thiết bị truyền âm thanh liên tục, 16 kHz, 16 bit
HZ = sach(16, trich="(16 kHz, 16-bit)") * 1000
BIT = sach(16, trich="(16 kHz, 16-bit)")
theo_sach("một thiết bị (KB/s)", HZ * BIT / 8 / 1e3, sach=32, trich="~32 KB/s", tol=0.5)
theo_sach("1 tỉ thiết bị (TB/s)", THIET_BI * HZ * BIT / 8 / 1e12, sach=32, trich="32 TB/s")

Một mô hình nhanh hơn có cứu được kịch bản này không? Hãy quét thời gian GPU cho mỗi truy vấn:

In [ ]:
for ms in (200, 100, 20):
    tb = truy_van_ngay * ms / 1e3 / 3600 / tt_gio
    print(f"{ms:>3} ms GPU mỗi truy vấn: {tb:5.2f} trung tâm dữ liệu trung bình, {DINH * tb:5.2f} lúc cao điểm")

Ngay cả khi mỗi truy vấn nhanh gấp 10 lần, lúc cao điểm vẫn cần khoảng hai trung tâm dữ liệu chỉ cho giọng
nói. Chương kết luận xử lý giọng nói hoàn toàn trên cloud không chỉ đắt, mà còn nặng nề về vận hành ở quy mô
toàn cầu. Lối ra là kiến trúc kết hợp: phát hiện từ đánh thức (wake word) chạy tại chỗ trên phần cứng TinyML, nên âm
thanh không rời thiết bị cho tới khi người dùng thật sự gọi, và chỉ phần hiểu ngôn ngữ mới lên cloud.

## Thử thêm

1. Ở phần 1, chọn hai thành phố bạn biết, tra khoảng cách đường thẳng giữa chúng, và tính mức sàn một
   vòng. Một trò chơi trực tuyến cần phản hồi dưới 16.7 ms có đặt máy chủ ở thành phố kia được không?
2. Ở phần 2, giả sử drone nén mỗi khung xuống 100 lần trước khi gửi. Đường xa có kịp một khung hình không?
   Phần nào của công thức bây giờ chi phối? (Các dấu ✗ khi đổi tham số là điều được chờ đợi.)
3. Ở phần 4, đổi chi phí radio thành 10 mJ mỗi MB, rồi 1,000 mJ mỗi MB. Cỡ dữ liệu hoà vốn dịch đi thế
   nào, và kết luận của cảm biến động vật hoang dã có đổi không?

## Tóm lại

* Ánh sáng trong sợi quang đặt mức sàn 1 ms cho mỗi 100 km một chiều; ngân sách 10 ms loại mọi trung tâm
  dữ liệu xa hơn 1,000 km, trước cả khi tính tới mạng thật và mô hình.
* Bất biến cục bộ dữ liệu so toàn bộ đường xa với hạn chót: với dữ liệu nhỏ phần cố định chi phối, với dữ
  liệu lớn thời gian truyền chi phối, và tính toán ở xa nhanh hơn không xoá được cả hai.
* Khi tốc độ dữ liệu vượt đường truyền, như 100 camera thô so với một đường 10 Gbps, xử lý ở edge là bắt
  buộc; lọc tại chỗ giảm lưu lượng hàng trăm nghìn lần.
* Với thiết bị chạy pin, gửi một khung dữ liệu có thể tốn hơn tự tính hàng nghìn lần; chỉ dữ liệu cỡ bản
  tóm tắt mới đáng gửi.
* Ở quy mô hàng tỉ thiết bị, chính phép tính năng lực đẩy kiến trúc về dạng kết hợp, với tầng TinyML lọc
  trước khi cloud nhận việc.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Hệ thống học máy, bốn môi trường triển khai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch02-ml-systems/), dựng từ chương
[*ML Systems*](https://mlsysbook.ai/vol1/ml_systems/ml_systems.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_02_ml_systems.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.